# 匹配粒子，追溯粒子，示例

### 这个示例主要展示了两个部分：
##### 1. 如何使用AnastrisTNG中的IDFinder工具来追溯恒星或者暗物质粒子在不同快照中的位置和属性。
##### 2. 如何使用IDFinder工具，利用Tracer来追溯重子物质的质量来源（尤其对于气体粒子）。

In [1]:
import numpy as np
from AnastrisTNG import TNGsimulation
from AnastrisTNG.TNGtools import IDFinder

In [2]:
path = '/home/yxi/Simulation/sims/TNG50-1/output'
snap =99
snapshot = TNGsimulation.Snapshot(path,snap)

In [3]:
sub = snapshot.load_particle(706298)

### 一、追溯恒星或者暗物质粒子
##### 假设已知某个快照中某个子结构的粒子ID列表，想要追溯这些粒子在其他快照中的位置和属性。

In [4]:
target_id = sub.s["iord"][:1000] # 假设我们已经知道了某个子结构中粒子的ID列表，想要追溯这些粒子在其他快照中的位置和属性。

##### 对于恒星以及暗物质粒子，它们的ID在不同快照中是保持不变的，因此可以直接使用这些ID来追溯它们在其他快照中的位置和属性。

In [5]:
target_snap = 99 # 目标快照编号, 可以是任意快照编号

In [6]:
res = IDFinder(path, target_snap
         ).find(
        target_id, 
        id_field="PartType4/ParticleIDs",
        return_fields=["PartType4/Coordinates", "PartType4/Masses"],
        NP=4)
# id_field 是需要匹配的粒子ID所在的字段，"PartType4/ParticleIDs" 即恒星粒子的ID字段，
# 如果是暗物质粒子，则应该是 "PartType1/ParticleIDs"。
# return_fields 是需要返回的字段列表，可以根据需要调整，比如还可以返回速度、金属丰度等信息。
# NP 是并行处理的进程数，可以根据实际情况调整, 默认是1，设置为4可以加快处理速度。

  0%|          | 0/680 [00:00<?, ?it/s]

100%|██████████| 680/680 [00:49<00:00, 13.69it/s]


In [7]:
res.keys()

dict_keys(['PartType4/ParticleIDs', 'PartType4/Coordinates', 'PartType4/Masses'])

#### 返回的结果相应的在字典res中，键是字段名称，值是对应的数组。例如：

In [8]:
for i, item in res.items():
    print(f"{i}: => {np.shape(item)}")

PartType4/ParticleIDs: => (1000,)
PartType4/Coordinates: => (1000, 3)
PartType4/Masses: => (1000,)


##### 这样就完成了粒子的追溯和匹配，得到了目标快照中对应粒子的属性数据。

### 二、追溯气体粒子
##### 对于气体粒子(更严谨来说是cell)，由于它们所包含的物质在不同快照中可能会发生变化，因此需要使用Tracer来追溯它们的质量来源。
##### 因此该过程分为三步：
##### 1. 首先匹配目标快照中气体粒子对应的Tracer ID，这些Tracer ID是唯一的，可以用来追溯它们在其他快照中的位置。
##### 2. 然后使用这些Tracer ID来追溯它们在其他快照中的所在气体(也可能是恒星)粒子的ID
##### 3. 最后根据这些气体粒子的ID获取它们在其他快照中的位置和属性

In [9]:
target_id = sub.g["iord"][:1000] # 假设我们想要追溯气体粒子，那么我们需要使用Tracer来追溯它们的质量来源。

current_snap = 99 # 当前 target_id 所在的快照编号

In [11]:
#第一步，匹配目标快照中气体粒子对应的Tracer ID，这些Tracer ID是唯一的，可以用来追溯它们在其他快照中的位置。
current_trace = IDFinder(path, current_snap
         ).find_tracers(
        target_id,
        NP=4
         )

  0%|          | 0/680 [00:00<?, ?it/s]

100%|██████████| 680/680 [07:10<00:00,  1.58it/s]


In [12]:
current_trace

{'ParentID': array([153931458563, 152872034799, 153615709649, ..., 153551007581,
        153457988835, 153697983675], shape=(1140,)),
 'TracerID': array([107384231675, 107215101195, 107347353083, ..., 107347638011,
        107209672203, 107375973371], shape=(1140,))}

##### 返回结果ParentID 是气体粒子(cell)的ID， TracerID 是对应的Tracer ID
###### 需要注意的是，有些气体粒子(cell)可能没有被附着Tracer，有些气体粒子(cell)可能被多个Tracer附着。
###### 这是因为气体粒子(cell)是空间网格，Tracer是质量元素，有的气体粒子质量较大，可能会被多个Tracer附着，而有的气体粒子质量较小，可能没有被任何Tracer附着。

#### 然后进行第二步，根据这些Tracer ID来追溯它们在其他快照中的所在气体(也可能是恒星)粒子的ID

In [13]:
target_snap = 99 # 目标快照编号, 可以是任意快照编号， 此处以当前快照为例

In [14]:
tar_tracer = IDFinder(path, target_snap
         ).find_tracers(
        current_trace["TracerID"],
        istracerid=True,
        NP=4
        )
# 这里传入 已经找到的 tracer的ID, current_trace["TracerID"], 并且设置 istracerid=True 来告诉函数我们传入的是 tracer的ID 而不是粒子ID。

100%|██████████| 680/680 [09:41<00:00,  1.17it/s]


In [15]:
tar_tracer

{'ParentID': array([153931458563, 152872034799, 153615709649, ..., 153551007581,
        153457988835, 153697983675], shape=(1140,)),
 'TracerID': array([107384231675, 107215101195, 107347353083, ..., 107347638011,
        107209672203, 107375973371], shape=(1140,))}

##### 这样就完成了气体粒子的追溯和匹配，得到了目标快照中对应的粒子的ID（ParentID）
#### 然后进行第三步，根据这些粒子的ID来获取它们在目标快照中的位置和属性，方法与第一部分追溯恒星或者暗物质粒子类似，只需要将id_field改为对应的气体粒子ID字段即可。

In [16]:
prop = IDFinder(path, target_snap
         ).find(
        tar_tracer["ParentID"],
        id_field="PartType0/ParticleIDs",
        return_fields=["PartType0/Coordinates", "PartType0/Masses"],
        NP=4
         )
# PartType0 是气体粒子， ParentID 是气体粒子(cell)的ID， 通过这个ID我们可以获取它在目标快照中的位置和属性。

100%|██████████| 680/680 [06:27<00:00,  1.76it/s]


In [18]:
for i, item in prop.items():
    print(f"{i}: => {np.shape(item)}, [...{item[:5]}...]")

PartType0/ParticleIDs: => (682,), [...[150363371588 153857861000 153927807330 153860230201 153932775350]...]
PartType0/Coordinates: => (682, 3), [...[[24451.01750697   169.2837454  20602.39894829]
 [24450.99420474   169.23181908 20602.42121805]
 [24450.5091468    169.45270121 20602.43575883]
 [24450.91410083   169.24546023 20602.32588574]
 [24450.50819209   169.41944096 20602.50594718]]...]
PartType0/Masses: => (682,), [...[3.9814677e-06 8.6513164e-06 9.3180061e-06 5.9400786e-06 1.3726545e-05]...]


##### 可以看到追溯出来的粒子位置和属性数据。
###### 需要注意的是，即使追溯的快照是当前快照，返回的结果也可能不完全一样。
###### 这里我们追溯了1000个气体粒子，返回的结果是682个粒子，
###### 原因 如上所述，气体粒子与Tracer的关系是并不是一对一的。

#### Note: 当需要追溯多个星系或者多个子结构的粒子时，可以将它们的粒子ID数组合并成一个大的数组，然后进行统一的一次追溯。追溯完成后，可以根据返回结果中的粒子ID来区分它们属于哪个星系或者哪个子结构。这样比逐个追溯每个星系或者子结构的粒子更高效。